# Agilent Bravo quickstart

The Agilent Bravo is a fixed-head liquid handler: an interchangeable pipetting head moves over a fixed 3x3 grid of nine deck sites, picks up disposable tips, aspirates and dispenses, and -- on models with a gripper -- moves plates between sites. PyLabRobot drives it through `AgilentBravoBackend`, a `LiquidHandlerBackend` exported from `pylabrobot.agilent` alongside the `Bravo` device facade and the `BravoDeck` deck model.

| Property | Value |
|---|---|
| Darwin | Ethernet TCP, port 7613. Gripper: yes. |
| Agile 7612 | Ethernet TCP, port 7612. Gripper: yes. |
| Bravo SRT | Ethernet TCP, port 7612 (same wire protocol as Agile 7612). Gripper: no. |
| Legacy Agile | Serial, or Ethernet TCP, port 10000. Gripper: yes. |
| Simulation | `SimulationController` -- no hardware or transport needed. |
| Head geometries | 8x1 and 16x1 disposable-tip heads at 9 mm and 4.5 mm barrel pitch; 8x12 at 9 mm pitch; 16x24 at 4.5 mm pitch. `num_channels` is 8, 16, 96, or 384. |
| Deck | Fixed 3x3 grid of nine sites, positioned from the instrument's own taught teachpoints. |

```{note}
This driver is unverified against real Agilent Bravo hardware. The protocol and controller layers it drives (`pylabrobot.agilent.bravo.protocol`, `pylabrobot.agilent.bravo.controllers`, `pylabrobot.agilent.bravo.darwin`) are exercised against real Darwin, Agile 7612, Bravo SRT, and legacy Agile instruments. The PyLabRobot transport, the deck mapping (`pylabrobot.agilent.bravo.deck.resource`), and `AgilentBravoBackend` itself are not -- `AgilentBravoBackend.setup()` logs this warning every time it runs. Report anything you find at https://discuss.pylabrobot.org.
```


```{device-card} agilent-bravo
```


## How it talks

A Bravo speaks one of two unrelated binary protocols, depending on its controller generation:

- **Gemini** (Darwin) -- a framed TCP protocol. One request is outstanding at a time under a lock; a background thread reads frames off the wire and wakes whichever call is waiting on a response.
- **V11/Agile** (Agile 7612, Bravo SRT, legacy Agile) -- a length-prefixed framing that carries 10-byte Agile packets to one or two motor controllers over an internal bus (X/Y/Z/W on one, the gripper's G/Zg on the other, where present). The Agile 7612 generation swaps the frame's command and length fields and adds a CRC-8/MAXIM check the legacy Agile generation does not have.

Both protocols live entirely in `pylabrobot.agilent.bravo.protocol` and never open a connection themselves. A controller (`pylabrobot.agilent.bravo.controllers`, or `pylabrobot.agilent.bravo.darwin` for Gemini) drives one of them over a `pylabrobot.agilent.bravo.transport` connection you construct and hand it.


## Physical setup

Connect over the network for Darwin, Agile 7612, and Bravo SRT, or over serial (or TCP port 10000) for a legacy Agile.

```{warning}
The Bravo moves a pipetting head -- and, on gripper models, a plate gripper -- over a fixed deck under motor power. A wrong deck assignment, a bad teachpoint, or an untested protocol can break labware, crush the pipette head into a plate, or injure a hand in its path. Keep the workspace clear and the emergency stop reachable, and dry-run every protocol against `SimulationController` before running it against real hardware.
```


## Connect (simulation)

`SimulationController` needs no transport or hardware: every axis tracks its position in memory and starts already homed. Build a `Bravo` facade around it, wrap it in `AgilentBravoBackend`, and hand that and a `BravoDeck` to a PyLabRobot `LiquidHandler`.

`config.head.teach_tip_length_mm` records the measured length of the tip that was on the head when the deck's teachpoints were taught; liquid handling and gripper moves need it to compute a safe approach height.


In [ ]:
from pylabrobot.agilent import AgilentBravoBackend, Bravo
from pylabrobot.agilent.bravo.config import BravoMachineConfig
from pylabrobot.agilent.bravo.controllers.simulation import SimulationController
from pylabrobot.agilent.bravo.deck.resource import BravoDeck
from pylabrobot.legacy.liquid_handling import LiquidHandler

controller = SimulationController(head_type="96_d_70")
deck = BravoDeck(head_type="96_d_70")
config = BravoMachineConfig()
config.head.teach_tip_length_mm = 19.9  # measured length of the tips this example uses

bravo = Bravo(controller, config=config, deck=deck)
backend = AgilentBravoBackend(bravo)
lh = LiquidHandler(backend=backend, deck=deck)

await lh.setup()


## Connect (real hardware)

Real hardware follows the same shape, with a real transport and the controller matching the instrument's generation: `DarwinController` for Darwin (over a `SocketTransport` at port 7613), `Agile7612Controller` for the Agile 7612 generation, `AgileSrtController` for a gripperless Bravo SRT (same wire protocol, port 7612), or `AgileController` for a legacy Agile (`SerialTransport`, or `SocketTransport` at port 10000). This cell is not run in this notebook -- it needs a real instrument at `BRAVO_IP`.


In [ ]:
from pylabrobot.agilent import AgilentBravoBackend, Bravo
from pylabrobot.agilent.bravo.config import BravoMachineConfig
from pylabrobot.agilent.bravo.controllers.agile_7612 import Agile7612Controller
from pylabrobot.agilent.bravo.deck.resource import BravoDeck
from pylabrobot.agilent.bravo.transport.socket import SocketTransport
from pylabrobot.legacy.liquid_handling import LiquidHandler

BRAVO_IP = "192.168.0.10"  # Replace with this instrument's address.

transport = SocketTransport("bravo", BRAVO_IP, 7612)
controller = Agile7612Controller(transport)
deck = BravoDeck(head_type="96_d_70")
config = BravoMachineConfig()
config.head.teach_tip_length_mm = 19.9  # Replace with the taught tip's measured length.

bravo = Bravo(controller, transport=transport, config=config, deck=deck)
backend = AgilentBravoBackend(bravo)
lh = LiquidHandler(backend=backend, deck=deck)

await lh.setup()


## Home the machine

Homing moves every axis to its reference position and must run with the deck and head path clear. `Bravo.home()` is reached through `backend.bravo`, since homing is not part of PyLabRobot's own `LiquidHandler` interface.


In [ ]:
homed_axes = await backend.bravo.home()
homed_axes


## Deck and teachpoints

`BravoDeck` models the instrument's fixed 3x3 grid of nine sites, each positioned at the X/Y/Z the instrument was taught for it (`deck.teachpoints`). Assign labware to a site with `assign_child_at_site`; the backend pushes it into the driver's own labware model the first time an operation touches that site.


In [ ]:
from pylabrobot.resources import Trash, cor_96_wellplate_360uL_Fb, opentrons_96_tiprack_10ul

tip_rack = opentrons_96_tiprack_10ul(name="tip_rack_1")
deck.assign_child_at_site(tip_rack, 4)

assay_plate = cor_96_wellplate_360uL_Fb(name="assay_plate_1")
deck.assign_child_at_site(assay_plate, 5)

trash = Trash(name="trash_1", size_x=127.0, size_y=86.0, size_z=40.0)
deck.assign_child_at_site(trash, 9)

deck.teachpoints.get_teachpoint(5, "x"), deck.teachpoints.get_teachpoint(5, "y")


## Tip pickup and the rectangular-block rule

Every operation the Bravo head performs works on a contiguous rectangular block of barrels, anchored at one of the head's four corners: a single well, a full row or column, a rectangle, or the whole head. PyLabRobot lets you select any set of wells or tip spots, but the driver only accepts a selection whose identifiers form a complete rectangle -- a diagonal pair, an L-shape, or a rectangle with a hole in it is rejected, with an explanation of what would work instead.


In [ ]:
try:
  await lh.pick_up_tips(tip_rack["A6", "B7"])
except Exception as exc:
  print(f"{type(exc).__name__}: {exc}")


The smallest legal block is a single well.


In [ ]:
await lh.pick_up_tips(tip_rack["H12"])
backend.bravo.head_mode


Drop it before picking up the column this notebook uses next -- the head can only ever hold one block of tips at a time.


In [ ]:
await lh.drop_tips([trash])


A full column is a rectangular block too -- eight rows, one column. Note the colon inside the string, `"A1:H1"`: PyLabRobot's item-selection syntax is inclusive of both ends. The Python slice form, `tip_rack["A1":"H1"]`, is exclusive of the stop and would only select seven wells here.


In [ ]:
await lh.pick_up_tips(tip_rack["A1:H1"])
backend.bravo.head_mode


## Aspirate

Volume and flow rate are shared by every active channel in one call, since the whole head is driven by a single plunger.


In [ ]:
await lh.aspirate(assay_plate["A1:H1"], vols=[5.0] * 8)


## Dispense


In [ ]:
await lh.dispense(assay_plate["A1:H1"], vols=[5.0] * 8)


## Tip drop

Drop to a tip box position to return tips, or to a `Trash` resource to discard them; either way, every channel in one call must target the same kind of resource. This notebook drops to trash: a tip box tracks its own depletion in full row/column bands, so only a block that spans the box's full width or height (a `row`, `column`, or `all_barrels` head mode) returns cleanly. A `single_barrel` or narrow `rectangle` return next to still-occupied neighbours is rejected -- even back to the exact cell a tip came from -- with an explanation of which head modes do round-trip.


In [ ]:
await lh.drop_tips([trash] * 8)


## Move a plate with the gripper

`LiquidHandler.move_plate()` is the call a PyLabRobot user actually makes to move a plate. `AgilentBravoBackend` reports `num_arms=1` on a gripper-equipped model, so PyLabRobot routes the call through this backend's `pick_up_resource`/`move_picked_up_resource`/`drop_resource` as normal. On a gripperless Bravo SRT, the backend reports `num_arms=0` and the same call raises PyLabRobot's own `"No robotic arm is installed on this liquid handler."` before ever reaching the backend -- the correct layering, not a Bravo-specific failure.


In [ ]:
target_site = deck.get_resource(f"{deck.name}_site_6")
await lh.move_plate(assay_plate, target_site)
deck.site_for_resource(assay_plate)


## Limits

- **384-channel heads** work through the per-channel path (`pick_up_tips`/`aspirate`/`dispense`/`drop_tips`) like any other head. The 96-head path (`pick_up_tips96`/`aspirate96`/`dispense96`/`drop_tips96`) is unreachable for one: PyLabRobot's own `LiquidHandler.pick_up_tips96` hard-requires exactly 96 populated tip positions before the backend is ever called.
- **The Bravo SRT has no gripper.** `AgilentBravoBackend.num_arms` is `0` for it, so `LiquidHandler.move_plate()`/`pick_up_resource()` raise PyLabRobot's own `"No robotic arm is installed on this liquid handler."` before ever reaching the backend; a caller that bypasses `LiquidHandler` and calls `pick_up_resource`/`move_picked_up_resource`/`drop_resource` directly hits this backend's own gripper check instead.
- **`BravoDeck`'s well-grid sign convention is unconfirmed against real hardware.** The offset from a site's teachpoint to well A1 is taken directly from the resource's own item-grid metadata, un-negated; whether that matches where the instrument physically expects A1 has not been checked on a real Bravo. Under `BravoDeck`'s default teachpoints for a `96_d_70` head and a `cor_96_wellplate_360uL_Fb`, this leaves only deck sites 5 and 8 with every one of the plate's 96 wells reachable as a plate anchor -- sites 1, 2, and 3 have none reachable at all, and sites 4, 6, 7, and 9 have some but not all. See `pylabrobot/agilent/bravo/deck/resource.py` for the details.


## Disconnect


In [ ]:
await lh.stop()
